In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

class AsistenteMetroPro:
    def __init__(self):
        load_dotenv()
        self.client = OpenAI(
            base_url=os.environ.get("GITHUB_BASE_URL"),
            api_key=os.environ.get("GITHUB_TOKEN")
        )

    def analizar_viaje_estructurado(self, consulta_usuario):
        # ANATOMÍA DEL PROMPT (Contexto + Tarea + Ejemplos + Formato + Restricciones)
        # Basado en la estructura de los cuadernos IL1.2
        prompt_sistema = """
        Eres un analista experto de transporte para Metro de Santiago.
        
        CONTEXTO:
        - Horario Punta (07:00-08:59, 18:00-19:59): $840
        - Horario Valle: $760 | Horario Bajo: $680
        
        TAREA: 
        Extraer los datos del viaje del usuario y devolver un JSON estructurado.
        
        EJEMPLOS (Few-Shot Prompting):
        User: "Voy de Tobalaba a Los Héroes, son las 8:15 am"
        Assistant: {
            "metacognicion": "El usuario viaja en la mañana dentro del rango Punta. La ruta es directa por L1.",
            "origen": "Tobalaba",
            "destino": "Los Héroes",
            "horario": "Punta",
            "tarifa": 840,
            "combinacion": "Ninguna"
        }
        
        RESTRICCIONES:
        1. Responde UNICAMENTE con el objeto JSON.
        2. Si falta el horario, asume 'Valle' por defecto.
        3. La clave 'metacognicion' debe explicar el razonamiento lógico del cálculo.
        """

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": prompt_sistema},
                    {"role": "user", "content": consulta_usuario}
                ],
                # Forzamos que la salida sea JSON (Característica avanzada)
                response_format={ "type": "json_object" }
            )
            
            # Parseamos para mostrar que podemos usar los datos en una base de datos
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            return {"error": f"Fallo en el servidor: {str(e)}"}

# === PRUEBA PARA LA CLASE ===
if __name__ == "__main__":
    asistente = AsistenteMetroPro()
    pregunta = "Necesito ir desde Plaza Puente Alto hasta Pedro de Valdivia a las 18:30"
    
    print(f"PROCESANDO CONSULTA: {pregunta}...\n")
    resultado = asistente.analizar_viaje_estructurado(pregunta)
    
    # Imprimimos el JSON bien formateado
    print("=== DATOS ESTRUCTURADOS (LISTOS PARA DB) ===")
    print(json.dumps(resultado, indent=4, ensure_ascii=False))